# Import Libraries

In [ ]:
from langchain.agents.middleware import dynamic_prompt,ModelRequest
from langchain.agents.middleware import SummarizationMiddleware
from langgraph.checkpoint.postgres import PostgresSaver  
from langgraph.store.postgres import PostgresStore 
from langchain.tools import tool,ToolRuntime
from dataclasses import dataclass

from langchain.agents import create_agent
from langchain_groq import ChatGroq
from dotenv import load_dotenv
import os 

DB_URI = os.getenv("DATABASE_URL")
# DB_URI

# Tools

In [70]:
@dataclass
class UserContext:
    user_id: str
    tier: str

@tool
def count_messages(runtime: ToolRuntime[UserContext]) -> int:
    "Count the number of messages in the current conversation"
    return len(runtime.state.get("messages"))

@tool
def save_user_memory(name: str,language:str,runtime: ToolRuntime[UserContext])-> str:
    "Save user name and preferred programming language to memory"
    runtime.store.put(
        ("users",),
        runtime.context.user_id,
        {"name":name,"language":language}
    )

    return "Saved user memory successfully "

@tool
def get_user_memory(runtime: ToolRuntime[UserContext])-> str:
    "Get user name and preferred programming language from memory"
    item = runtime.store.get(("users",),runtime.context.user_id)
    return str(item.value) if item else "No user memory found"



# Initialize LLM

In [71]:
load_dotenv()

model = ChatGroq(
    model="openai/gpt-oss-120b"
)

# Create Middleware

In [ ]:


@dynamic_prompt
def dynamic_prompt_middleware(request: ModelRequest) -> ModelRequest:
    ctx = request.runtime.context
    style = "concise and technical" if ctx.tier == "premium" else "simple and easy to understand"
    return f"You are a helpful assistant.You are assisting user {ctx.user_id}.Keep responses{style}"


# Run Agent

In [ ]:

with (
    PostgresSaver.from_conn_string(DB_URI) as checkpointer, 
    PostgresStore.from_conn_string(DB_URI) as store):
    checkpointer.setup() # auto create tables in PostgreSQL
    store.setup() # auto create tables in PostgreSQL
    agent = create_agent(
        model=model,
        tools=[save_user_memory,get_user_memory],
        checkpointer=checkpointer,
        store=store,
        middleware=[dynamic_prompt_middleware,
                    SummarizationMiddleware(
                        model=model,
                        trigger={"max_tokens": 1000},
                        keep={"max_tokens": 500}
                    )],
        context_schema=UserContext,
        system_prompt="You are a helpful assistant that can remember user information and retrieve it when asked."
    )

    response = agent.invoke(
        {"messages":[{"role":"user","content":"What is async/await in python"}]},
        config={"configurable":{"thread_id": "11"}},
        context=UserContext(user_id="user_101", tier="free")
    )

    print(response["messages"][-1].content)
    print(len(response["messages"]))

**Async/await** is Python’s way of writing *asynchronous* code—code that can do other work while it’s waiting for something (like a file read, a web request, or a timer) instead of stopping everything.

---

### The key ideas

| Concept | What it means |
|---------|---------------|
| **Coroutine** | A special function declared with `async def`. It can pause and resume, letting other code run in the meantime. |
| **`await`** | Inside a coroutine you write `await <awaitable>`. It tells Python: “Pause here until this awaitable is finished, then continue.” |
| **Event loop** | The engine that drives coroutines. It keeps track of which coroutine is ready to run and switches between them. |
| **Awaitable** | Anything you can `await`: another coroutine, an `asyncio` task, or an object that implements `__await__`. |

---

### Simple example

```python
import asyncio          # built‑in library that provides the event loop

async def say_after(delay, message):
    await asyncio.sleep(delay)   #